# Типы признаков и apply на заказах


Работаем как команда CRM маркетплейса: из трёх связанных таблиц нужно получить
объяснимые признаки, а не просто добиться вывода без ошибки. Перед каждой
операцией сформулируйте единицу наблюдения, ключ соединения и ожидаемое число
строк. После операции прочитайте assert как исполняемый контракт.

Сначала сделайте минимальный рабочий вариант, затем проверьте его на данных и
только после этого интерпретируйте результат. Не вводите метку churn: в этом
модуле мы строим и проверяем признаки, но не обучаем модель оттока.


**Центральная идея:** Тип признака определяет допустимое преобразование; ключ и форма таблицы определяют корректность join.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

def find_csv(name):
    for path in (
        Path(name),
        Path("../") / name,
        Path("../../data") / name,
        Path("../data") / name,
        Path("../../../data") / name,
    ):
        if path.exists():
            return path.resolve()
    return "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_05_shop_feature_engineering/data/" + name

orders = pd.read_csv(find_csv("orders_slim.csv"), parse_dates=["order_purchase_timestamp", "order_delivered_customer_date"])
customers = pd.read_csv(find_csv("customers_slim.csv"))
payments = pd.read_csv(find_csv("payments_slim.csv"))
assert len(orders) and len(customers) and len(payments)
assert orders["order_id"].is_unique and customers["customer_id"].is_unique
print(f"orders={len(orders)}, customers={len(customers)}, payments={len(payments)}")


## 1. Три таблицы — три единицы наблюдения

Посчитайте строки и назовите единицу наблюдения каждой таблицы.

**Зачем:** Тип признака определяет допустимое преобразование; ключ и форма таблицы определяют корректность join. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
sizes = {}  # TODO: orders/customers/payments -> len
assert sizes == {"orders": 3500, "customers": 778, "payments": 3500}
UNITS_NOTE = ""  # TODO: не менее 140 символов
assert len(UNITS_NOTE) >= 140


## 2. Числовые и категориальные столбцы

Соберите словарь типов вручную для четырёх бизнес-полей.

**Зачем:** Тип признака определяет допустимое преобразование; ключ и форма таблицы определяют корректность join. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
feature_types = {"payment_value": None, "payment_type": None, "customer_state": None, "order_status": None}  # TODO
assert feature_types["payment_value"] == "numeric"
assert set(feature_types.values()) == {"numeric", "categorical"}


## 3. Безопасный join заказов и оплат

Соедините по order_id слева и докажите сохранение строк.

**Зачем:** Тип признака определяет допустимое преобразование; ключ и форма таблицы определяют корректность join. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
orders_pay = None  # TODO
assert isinstance(orders_pay, pd.DataFrame) and len(orders_pay) == len(orders)
assert {"payment_type", "payment_value"} <= set(orders_pay)
assert orders_pay["order_id"].is_unique


## 4. Функция категории оплаты

Напишите именованную функцию amount_band и проверьте границы 100 и 300.

**Зачем:** Тип признака определяет допустимое преобразование; ключ и форма таблицы определяют корректность join. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
def amount_band(value):
    # TODO: small <=100, mid <=300, иначе big
    ...
assert [amount_band(x) for x in (50, 100, 101, 300, 301)] == ["small","small","mid","mid","big"]


## 5. Apply к одному столбцу

Примените amount_band и проверьте покрытие всех строк.

**Зачем:** Тип признака определяет допустимое преобразование; ключ и форма таблицы определяют корректность join. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
orders_pay["payment_band"] = None  # TODO
band_counts = None  # TODO
assert set(orders_pay["payment_band"]) == {"small", "mid", "big"}
assert int(band_counts.sum()) == len(orders_pay)


## 6. Apply по строке

Рассчитайте days_to_deliver из двух дат с обработкой недоставленных заказов.

**Зачем:** Тип признака определяет допустимое преобразование; ключ и форма таблицы определяют корректность join. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
orders_pay["days_to_deliver"] = None  # TODO: apply(axis=1)
assert orders_pay["days_to_deliver"].notna().sum() > 3000
assert orders_pay["days_to_deliver"].dropna().ge(0).all()


## 7. Apply или векторизация

Повторите расчёт векторно и сравните совпадение.

**Зачем:** Тип признака определяет допустимое преобразование; ключ и форма таблицы определяют корректность join. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
vector_days = None  # TODO
same_days = None  # TODO
assert same_days is True
VECTOR_NOTE = ""  # TODO: минимум 160 символов
assert len(VECTOR_NOTE) >= 160


## 8. Карточка признака

Опишите источник, тип, правило, пропуски и риск для days_to_deliver.

**Зачем:** Тип признака определяет допустимое преобразование; ключ и форма таблицы определяют корректность join. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
FEATURE_CARD = ""  # TODO: минимум 240 символов
assert len(FEATURE_CARD) >= 240
assert all(word in FEATURE_CARD.lower() for word in ["источник", "пропуск", "риск"])
